# Maia Lite 0.3.0 — Scientific SFT LoRA pilot v1

**Objetivo:** ajustar somente adaptadores LoRA sobre o checkpoint original do step 300.000, com perda apenas nos tokens da resposta, validação por artigo e checkpoints separados. **Não modifica `latest.pt`.**

**Pré-requisitos:** Colab A100 80 GB, Drive montado, `sft_train.jsonl` e `sft_validation.jsonl` gerados pelo notebook de auditoria. Verifique direitos/licenças e alertas científicos antes de executar o treino. Os 273 exemplos de validação são internos, não um teste independente. O notebook é uma implementação experimental, não foi executado numa A100 nesta sessão.

**Pasta de entrada:** `MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/scientific_sft/prepared_v1/`.

**Saída:** `MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_scientific_sft_lora_v1/`.

In [23]:
import os, json, math, time, random, hashlib, gc, csv
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer
from tqdm.auto import tqdm

SEED=20261118
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Este piloto requer GPU CUDA; selecione A100 no Colab.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:',torch.__version__)
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: confirme os caminhos.')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive não montado.'


GPU: NVIDIA L4 | Torch: 2.14.1+cu130


In [24]:
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
PREP=ROOT/'datasets/scientific_sft/prepared_v1'
TRAIN_FILE=PREP/'sft_train.jsonl'
VAL_FILE=PREP/'sft_validation.jsonl'
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
OUT = ROOT/'training_runs/maia_lite_0_3_0_scientific_sft_lora_L4_v1'
for path in (TRAIN_FILE,VAL_FILE,BASE_CKPT): assert path.is_file(),f'Arquivo ausente: {path}'
assert TOKENIZER_DIR.is_dir(),f'Tokenizer ausente: {TOKENIZER_DIR}'
for path in (TRAIN_FILE,VAL_FILE,BASE_CKPT): print(path, path.stat().st_size)
print('Saída:',OUT)


/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/scientific_sft/prepared_v1/sft_train.jsonl 1285183
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/scientific_sft/prepared_v1/sft_validation.jsonl 143970
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt 6084855785
Saída: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_scientific_sft_lora_L4_v1


## Configuração do piloto

Comece com **1 época**. Revise as métricas antes de alterar `EPOCHS` para 2 ou 3. A avaliação é feita por perda teacher-forced; respostas livres são testadas numa célula separada. `MAX_LENGTH=512` é um limite inicial de custo, menor que o contexto nativo de 2048; registros longos serão **excluídos**, nunca truncados silenciosamente. A implementação LoRA abaixo atua em `qkv` e `proj` de cada bloco.

In [25]:
RANK=16; ALPHA=32; LORA_DROPOUT=0.05
LR=8e-5; WEIGHT_DECAY=0.0
MAX_LENGTH=512; EPOCHS=1; MICRO_BATCH=1; GRAD_ACCUM=16
MAX_GRAD_NORM=1.0; VALIDATE_EVERY=50; SAVE_EVERY=100
MAX_STEPS=None   # limite opcional para teste rápido (ex.: 10)
RUN_TRAINING=True  # ALTERE PARA True APÓS REVISAR A AUDITORIA E O PREVIEW
assert MAX_LENGTH<=2048


In [26]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## Carregar checkpoint original (somente leitura)

`weights_only=False` é necessário porque o checkpoint confiável do próprio treinamento contém estados RNG e optimizer. **Nunca use essa opção com checkpoints de origem desconhecida.** A arquitetura é reproduzida diretamente do notebook original v3.8.

In [27]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size,(len(tokenizer),cfg.vocab_size)
assert tokenizer.eos_token_id is not None,'Tokenizer sem EOS.'
print('EOS:',tokenizer.eos_token_id,'vocab:',len(tokenizer))
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False)
assert 'model' in checkpoint and 'step' in checkpoint, 'Formato de checkpoint inesperado.'
print('Step do checkpoint:',checkpoint['step'])
assert int(checkpoint['step'])==300000, 'Checkpoint não é step 300000: revise antes de continuar.'
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint; gc.collect()
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)
model.eval()
print('Modelo carregado, pesos base congelados.')


EOS: 2 vocab: 50257
Step do checkpoint: 300000
Modelo carregado, pesos base congelados.


In [28]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
trainable=[p for p in model.parameters() if p.requires_grad]
print('LoRA trainable parameters:',sum(p.numel() for p in trainable))
assert len(trainable)==cfg.n_layer*2*2


LoRA trainable parameters: 3538944


## Preparar sequências e máscara de perda

Formato fixo: `prompt` + `completion` + EOS. Apenas a resposta e o EOS recebem perda. A tokenização é feita sobre a sequência **completa**, para preservar merges BPE na fronteira prompt/resposta; a fronteira é estimada por `offset_mapping`. Amostras com fronteira ambígua são descartadas e relatadas.

In [29]:
def read_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
train_raw=read_jsonl(TRAIN_FILE); val_raw=read_jsonl(VAL_FILE)
assert len(train_raw)==2439 and len(val_raw)==273, (len(train_raw),len(val_raw))
assert not ({r['paperId'] for r in train_raw}&{r['paperId'] for r in val_raw})
assert not ({r['groupId'] for r in train_raw}&{r['groupId'] for r in val_raw})
print('Split sem sobreposição de paperId/groupId.')

def encode_record(rec):
    prompt=rec['prompt']; answer=rec['completion']
    if not prompt.strip() or not answer.strip():return None,'empty'
    text=prompt+answer
    enc=tokenizer(text,add_special_tokens=False,return_offsets_mapping=True)
    ids=enc['input_ids']; offsets=enc['offset_mapping']; boundary=len(prompt)
    # Reject BPE tokens that cross the prompt-answer boundary.
    if any(a<boundary<b for a,b in offsets):return None,'boundary_merge'
    answer_idx=next((i for i,(a,b) in enumerate(offsets) if a>=boundary and b>a),None)
    if answer_idx is None:return None,'no_answer_tokens'
    ids=ids+[tokenizer.eos_token_id]
    if len(ids)>MAX_LENGTH:return None,'too_long'
    # x=ids[:-1], labels predict ids[1:]; positions prior to answer are masked.
    labels=[-100]*(len(ids)-1)
    for i in range(len(labels)):
        if i+1>=answer_idx:labels[i]=ids[i+1]
    if all(v==-100 for v in labels):return None,'empty_targets'
    return {'x':ids[:-1],'y':labels,'language':rec['language'],'id':rec['id']},None

def prepare(rows):
    good=[]; dropped={}
    for r in rows:
        item,why=encode_record(r)
        if item is not None:good.append(item)
        else:dropped[why]=dropped.get(why,0)+1
    return good,dropped
train_data,train_drop=prepare(train_raw);val_data,val_drop=prepare(val_raw)
print('Train:',len(train_data),'excluídos:',train_drop)
print('Val:',len(val_data),'excluídos:',val_drop)
assert train_data and val_data
for d in (train_data,val_data):
    print('Idiomas:',{lang:sum(r['language']==lang for r in d) for lang in ['pt','en','es']})


Split sem sobreposição de paperId/groupId.
Train: 2439 excluídos: {}
Val: 273 excluídos: {}
Idiomas: {'pt': 813, 'en': 813, 'es': 813}
Idiomas: {'pt': 91, 'en': 91, 'es': 91}


In [30]:
def collate(batch):
    L=max(len(r['x']) for r in batch)
    x=torch.full((len(batch),L),tokenizer.eos_token_id,dtype=torch.long)
    y=torch.full((len(batch),L),-100,dtype=torch.long)
    for i,r in enumerate(batch):
        l=len(r['x']);x[i,:l]=torch.tensor(r['x']);y[i,:l]=torch.tensor(r['y'])
    return x.to('cuda'),y.to('cuda')

sample=train_data[0]
print('Exemplo:',sample['id'],'| tokens:',len(sample['x']))
print('Tokens supervisionados:',sum(z!=-100 for z in sample['y']))
print('Resposta supervisionada (preview):',tokenizer.decode([z for z in sample['y'] if z!=-100])[:500])


Exemplo: W4385245566-q01::en | tokens: 87
Tokens supervisionados: 40
Resposta supervisionada (preview): Over 75% of Mizar toplevel lemmas can be proved by AI/TP systems when the premises for the proof can be selected from the library either by a human or a machine.</s>


## Avaliação e checkpoints LoRA

Os arquivos salvos contêm **somente os adaptadores**, optimizer, scheduler e RNG, nunca o checkpoint base. Retomada automática de época com ordem determinística por seed e época. **A retomada é por passo de atualização**, e pressupõe os mesmos dados e hiperparâmetros. O checkpoint é salvo por substituição atômica de arquivo local temporário no mesmo diretório Drive; confirme o arquivo após a execução.

In [31]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
DATA_HASHES={'train':sha256(TRAIN_FILE),'validation':sha256(VAL_FILE)}

def lora_state():
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items() if '.A' in k or '.B' in k}

def restore_lora(state):
    keys={k for k in model.state_dict() if '.A' in k or '.B' in k}
    assert set(state)==keys,(len(state),len(keys))
    with torch.no_grad():
        for name,param in model.named_parameters():
            if name in state:param.copy_(state[name].to(param.device,dtype=param.dtype))

@torch.no_grad()
def evaluate():
    model.eval(); sums={}; counts={}
    for lang in ['pt','en','es']:
        sums[lang]=0.;counts[lang]=0
    for start in range(0,len(val_data),MICRO_BATCH):
        batch=val_data[start:start+MICRO_BATCH]
        x,y=collate(batch)
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
        losses=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='none').reshape_as(y)
        for i,r in enumerate(batch):
            mask=y[i]!=-100;lang=r['language']
            sums[lang]+=float(losses[i][mask].sum());counts[lang]+=int(mask.sum())
        del x,y,logits,losses
    result={f'val_loss_{k}':sums[k]/max(counts[k],1) for k in sums}
    result['val_loss']=sum(sums.values())/max(sum(counts.values()),1)
    model.train();return result

optimizer=torch.optim.AdamW(trainable,lr=LR,weight_decay=WEIGHT_DECAY)
updates_per_epoch=math.ceil(math.ceil(len(train_data)/MICRO_BATCH)/GRAD_ACCUM)
planned_updates=updates_per_epoch*EPOCHS
if MAX_STEPS is not None:planned_updates=min(planned_updates,MAX_STEPS)
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s: max(0.1,1-s/max(planned_updates,1)))
print('Updates/epoch:',updates_per_epoch,'planned:',planned_updates)


Updates/epoch: 153 planned: 153


In [32]:
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'rank':RANK,'alpha':ALPHA,'dropout':LORA_DROPOUT,'lr':LR,'max_length':MAX_LENGTH,
        'micro_batch':MICRO_BATCH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'max_steps':MAX_STEPS,
        'seed':SEED,'data_hashes':DATA_HASHES,'checkpoint_path':str(BASE_CKPT),'base_step':300000,
        'target_modules':['qkv','proj'],'architecture':'MaiaGPT v3.8'}
config_file=OUT/'config.json'
if config_file.exists():
    old=json.loads(config_file.read_text())
    assert old==CONFIG,'Configuração mudou em relação ao experimento existente. Use outro OUT.'
else:config_file.write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False))

ckpt_file=OUT/'adapter_latest.pt';start_update=0
if ckpt_file.exists():
    saved=torch.load(ckpt_file,map_location='cpu',weights_only=False)
    assert saved['config']==CONFIG
    restore_lora(saved['adapter']);optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
    start_update=saved['update'];random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng']);torch.set_rng_state(saved['torch_rng'])
    if torch.cuda.is_available():torch.cuda.set_rng_state_all(saved['cuda_rng'])
    print('Retomada do update:',start_update)
else:print('Novo experimento, nenhum adaptador anterior.')

def save_adapter(update,metrics=None):
    state={'adapter':lora_state(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
           'update':update,'config':CONFIG,'metrics':metrics,
           'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=OUT/'adapter_latest.tmp.pt'
    torch.save(state,tmp);os.replace(tmp,ckpt_file)
    print('Adaptador salvo:',ckpt_file,'update:',update)


Novo experimento, nenhum adaptador anterior.


## Executar piloto

Primeiro rode com `RUN_TRAINING=False` para verificar a preparação e a avaliação inicial. Depois altere para `True` e execute a célula abaixo. Não altere hiperparâmetros no meio de uma retomada. O loop calcula a perda ponderada por **tokens supervisionados** (não pela quantidade de padding).

In [33]:
if not RUN_TRAINING:
    print('Treinamento desativado. Pré-avaliação:',evaluate())
else:
    assert start_update<=planned_updates
    model.train(); optimizer.zero_grad(set_to_none=True)
    update=start_update
    progress=tqdm(total=planned_updates,initial=update,desc='SFT LoRA')
    log_path=OUT/'metrics.jsonl'
    # Deterministic epoch permutations and batch boundaries for restart at update boundaries.
    try:
        for epoch in range(EPOCHS):
            if update>=planned_updates:break
            order=np.random.default_rng(SEED+epoch).permutation(len(train_data)).tolist()
            micro_batches=[order[i:i+MICRO_BATCH] for i in range(0,len(order),MICRO_BATCH)]
            n_updates=math.ceil(len(micro_batches)/GRAD_ACCUM)
            for local_update in range(n_updates):
                global_update=epoch*updates_per_epoch+local_update
                if global_update<update:continue
                if update>=planned_updates:break
                indices=micro_batches[local_update*GRAD_ACCUM:(local_update+1)*GRAD_ACCUM]
                # Normalize accumulated token losses by the total supervised tokens in the update.
                total_tokens=sum(sum(z!=-100 for z in train_data[j]['y']) for ids in indices for j in ids)
                accumulated_loss=0.
                for ids in indices:
                    x,y=collate([train_data[j] for j in ids])
                    with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
                    loss_sum=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='sum')
                    (loss_sum/max(total_tokens,1)).backward()
                    accumulated_loss+=float(loss_sum.detach())
                    del x,y,logits,loss_sum
                grad_norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
                optimizer.step();scheduler.step();optimizer.zero_grad(set_to_none=True)
                update+=1;progress.update(1)
                metrics={'update':update,'epoch':epoch+1,'train_loss':accumulated_loss/max(total_tokens,1),
                         'grad_norm':float(grad_norm),'lr':optimizer.param_groups[0]['lr']}
                if update%VALIDATE_EVERY==0 or update==planned_updates:
                    metrics.update(evaluate());print(metrics)
                with open(log_path,'a',encoding='utf-8') as f:f.write(json.dumps(metrics)+'\n')
                if update%SAVE_EVERY==0 or update==planned_updates:save_adapter(update,metrics)
    except KeyboardInterrupt:
        optimizer.zero_grad(set_to_none=True)
        print('Interrupção: somente atualizações completas são retomáveis. Último update:',update)
        save_adapter(update)
        raise
    finally:progress.close()
    print('Treino concluído até update',update)


SFT LoRA:   0%|          | 0/153 [00:00<?, ?it/s]

{'update': 50, 'epoch': 1, 'train_loss': 3.4278878391784815, 'grad_norm': 0.8749257326126099, 'lr': 5.3856209150326805e-05, 'val_loss_pt': 2.972011369853427, 'val_loss_en': 3.237651051108538, 'val_loss_es': 2.920844318814818, 'val_loss': 3.0336672589655933}
{'update': 100, 'epoch': 1, 'train_loss': 2.485961750097442, 'grad_norm': 0.7430466413497925, 'lr': 2.7712418300653598e-05, 'val_loss_pt': 2.940303318762366, 'val_loss_en': 3.2085038898782368, 'val_loss_es': 2.8877410538078445, 'val_loss': 3.002235202570462}
Adaptador salvo: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_scientific_sft_lora_L4_v1/adapter_latest.pt update: 100
{'update': 150, 'epoch': 1, 'train_loss': 3.028611760129552, 'grad_norm': 0.9406340718269348, 'lr': 8.000000000000001e-06, 'val_loss_pt': 2.9329867908107827, 'val_loss_en': 3.2011292775472007, 'val_loss_es': 2.8803540783825263, 'val_loss': 2.994876374589636}
{'update': 153, 'epoch': 1, 'train_loss': 3.0547577826703183, 'gra

## Teste qualitativo de geração (antes/depois)

O modelo base não possui chat template. A geração usa o mesmo estilo de prompt do SFT e é **apenas diagnóstica**, não substitui avaliação independente. Para comparar base e adaptador, execute com LoRA ativo e, separadamente, desative os módulos LoRA ou carregue o modelo base em uma nova sessão.

In [34]:
@torch.no_grad()
def generate(prompt,max_new_tokens=100,temperature=0.7):
    model.eval()
    ids=tokenizer(prompt,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
    for _ in range(max_new_tokens):
        if ids.shape[1]>=cfg.block_size:break
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(ids)
        last=logits[:,-1,:].float()/max(temperature,1e-4)
        nxt=torch.multinomial(torch.softmax(last,dim=-1),1)
        if nxt.item()==tokenizer.eos_token_id:break
        ids=torch.cat([ids,nxt],dim=1)
    model.train()
    return tokenizer.decode(ids[0],skip_special_tokens=True)

for lang in ['pt','en','es']:
    r=next(r for r in val_raw if r['language']==lang)
    print('\n',lang,r['id'],'\n',generate(r['prompt'])[-1200:])



 pt W3110989801-q01::pt 
 Responda à pergunta de forma clara, objetiva e cientificamente correta. Se não souber, indique a incerteza.

Quais são as quatro novas mudanças paradigmáticas previstas para redes de comunicação sem fio 6G?
O 9G será substituído pelo 5G

 en W3110989801-q01::en 
 Answer the question clearly, concisely, and scientifically accurately. If unsure, acknowledge uncertainty.

What are the four new paradigm shifts envisioned for 6G wireless communication networks?
Increasingly high-speed and lower-demand traffic is anticipated as a consequence of the four new paradigm shifts envisioned for 6G wireless communication networks: High-speed traffic increases, which is expected to be able to reach up to 20Gbps per second; High-demand traffic increases, which aims to reach up to 40Gbps per second; Low-demand traffic increases, which is expected to focus on the highest-density traffic in the 5G spectrum, while remaining relatively

 es W3110989801-q01::es 
 Responde a la pre

## Próximos passos

1. Envie `config.json`, `metrics.jsonl` e o último relatório de validação (não é necessário enviar o adaptador grande).
2. Compare com o checkpoint base usando um **conjunto externo independente** de prompts abertos.
3. Se a perda de validação piorar ou houver degradação qualitativa, não aumente as épocas automaticamente.
4. Para uso futuro, o adaptador requer **a mesma arquitetura, tokenizer e checkpoint base**. Não o aplique a outro step sem validação.